## Imports

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product

---
## Fonction de simulation

In [5]:
def run_simulation(A, omega, E, rho, L, Nx=100, Nt=100, t_end=1):
    
    v     = np.sqrt(E/rho)    # vitesse des ondes [m/s]

    # Grille
    x  = np.linspace(0, L, Nx)
    dx = x[1] - x[0]
    dt = t_end / Nt

    # Stable behaviour
    CFL = v * dt / dx
    assert CFL <= 1.0, f"CFL={CFL:.3f} > 1 : schéma instable"

    # Conditions aux limites
    
    t_pulse = 2 * np.pi / omega 

    def u_right(t): 
        if t < t_pulse:
            return A* np.sin(omega * t)
        else:
            return 0.0 

    # Initial state, the entire beam is at rest
    u     = np.zeros(Nx)
    u_dot = np.zeros(Nx)
    u_2 = np.zeros(Nx)

    u_storage = np.zeros((Nt+1,Nx))
    u_xx_storage=np.zeros((Nt,Nx))
    delta_u_storage=np.zeros((Nt,Nx))
    u_storage[0]=u.copy()
    u_dot_storage=np.zeros((Nt,Nx))

    rows = []

    # Timing loop
    for n in range(Nt):
        tn = n * dt

        # second derivative in space 
        u_xx = np.zeros(Nx)
        u_xx[1:-1] = (u[2:] - 2.0*u[1:-1] + u[:-2]) / dx**2

        #Schéma centré
        u_new=np.zeros(Nx)
        u_new[1:-1]=(2*u[1:-1])-u_2[1:-1]+CFL**2*(u[2:]-2*u[1:-1]+u[:-2])

        # 3. Conditions aux limites
        u_new[0]  = 0
        u_new[-1] = u_right(tn)

        u_xx_storage[n]    = u_xx
        delta_u_storage[n] = u_new - u

        # Vitesse à t_n par différence centrée
        u_dot = np.zeros(Nx)
        #u_dot[1:-1] = (u_new[1:-1] - u_2[1:-1]) / (2.0 * dt)
        u_dot[1:-1] = (u[1:-1] - u_2[1:-1]) / (dt)


        # Avancement : décalage des niveaux temporels
        u_2 = u.copy()    # l'ancien u^n devient u^{n-1}
        u   = u_new       # le nouveau u^{n+1} devient u^n
        u_storage[n + 1] = u.copy()
        u_dot_storage[n]=u_dot
    
    # Snapshot dans le dataset
    for n in range(Nt-6):
        for j in range(1, Nx - 1):     # nœuds intérieurs
            rows.append({
                #Parameters
                "A":A,
                "omega":omega,
                "t":n*dt,
                "x":j,
                
                #Inputs
                "u_dot":u_dot_storage[n,j],    
                "u_xx":u_xx_storage[n,j],
                
                # Outputs
                #"delta_u"     : delta_u_storage[n,j],
                "delta_u"     : u_storage[n+5,j]-u_storage[n,j],    
                #"delta_u"     : u_storage[n+1,j]-u_storage[n,j],    

                #Verification
                "u":u_storage[n+1,j],
            })

    
    return pd.DataFrame(rows)

## Génération du dataset complet

In [6]:
all_dfs = []

#Parameters
E=1
rho=1
L=1

Nt=400
Nx=100
t_end=4

N=5
AMPLITUDES = np.linspace(0.005, 0.1, N).round(3).tolist()
PULSATIONS = np.linspace(7, 90, N).round(1).tolist()

n_sims=len(AMPLITUDES)*len(PULSATIONS)

# We compute each simulation
for idx, (A, w) in enumerate(product(AMPLITUDES, PULSATIONS), start=1):
    df_run = run_simulation(A=A, omega=w, E=E,rho=rho, L=L, Nt=Nt, Nx=Nx, t_end=t_end)
    all_dfs.append(df_run)

# We concatenate each simulation in the df which is the full dataset table
df = pd.concat(all_dfs, ignore_index=True)
print(df)


            A  omega     t   x     u_dot       u_xx   delta_u         u
0       0.005    7.0  0.00   1  0.000000   0.000000  0.000000  0.000000
1       0.005    7.0  0.00   2  0.000000   0.000000  0.000000  0.000000
2       0.005    7.0  0.00   3  0.000000   0.000000  0.000000  0.000000
3       0.005    7.0  0.00   4  0.000000   0.000000  0.000000  0.000000
4       0.005    7.0  0.00   5  0.000000   0.000000  0.000000  0.000000
...       ...    ...   ...  ..       ...        ...       ...       ...
965295  0.100   90.0  3.93  94  0.058076 -18.026603 -0.077050 -0.000904
965296  0.100   90.0  3.93  95 -0.051485  11.095927 -0.044363  0.000302
965297  0.100   90.0  3.93  96  0.040177  -8.875741 -0.016496 -0.000258
965298  0.100   90.0  3.93  97 -0.027544   6.078138 -0.003406  0.000176
965299  0.100   90.0  3.93  98  0.014003  -3.087996 -0.000591 -0.000089

[965300 rows x 8 columns]



## Split train / val / test

In [7]:
rng = np.random.default_rng(seed=42)

combos     = list(product(AMPLITUDES, PULSATIONS))
combos_arr = rng.permutation(combos)

n_train = int(0.70 * n_sims)
n_val   = int(0.15 * n_sims)

train_combos = set(map(tuple, combos_arr[:n_train]))
val_combos   = set(map(tuple, combos_arr[n_train:n_train + n_val]))
test_combos  = set(map(tuple, combos_arr[n_train + n_val:]))

def assign_split(row):
    key = (row["A"], row["omega"])
    if key in train_combos: return "train"
    if key in val_combos:   return "val"
    return "test"

df["split"] = df.apply(assign_split, axis=1)

print("Distribution du split :")
for s in ["train", "val", "test"]:
    n = (df["split"] == s).sum()
    print(f"  {s:5s} : {n:>8,} lignes  ({100*n/len(df):.1f} %)")


Distribution du split :
  train :  656,404 lignes  (68.0 %)
  val   :  115,836 lignes  (12.0 %)
  test  :  193,060 lignes  (20.0 %)


## Normalisation (z-score)

In [8]:
INPUTS  = ["u_dot","u_xx"]
OUTPUTS = ["delta_u"]

# Masque train pour calculer les stats uniquement sur le train set
train_mask = df["split"] == "train"

# Calcul des stats sur le train set uniquement
norm_stats = pd.DataFrame({
    "mean": df.loc[train_mask, INPUTS + OUTPUTS].mean(),
    "std" : df.loc[train_mask, INPUTS + OUTPUTS].std(),
})
norm_stats["std"] = norm_stats["std"].replace(0, 1)  # éviter division par 0

# Application à tout le dataset
for col in INPUTS + OUTPUTS:
    df[col + "_n"] = (df[col] - norm_stats.loc[col, "mean"]) / norm_stats.loc[col, "std"]

print("Statistiques de normalisation (calculées sur train) :")
print(norm_stats.round(6))

Statistiques de normalisation (calculées sur train) :
             mean        std
u_dot    0.000002   0.715410
u_xx     0.000010  62.738980
delta_u -0.000002   0.021273


---
## Sauvegarde

In [9]:
COLS_ORDER = (
    ["A","omega"]
    +["t","x"]
    + INPUTS                                   # inputs bruts
    + [c + "_n" for c in INPUTS]             # inputs normalisés
    + OUTPUTS                                # outputs bruts
    + [c + "_n" for c in OUTPUTS]            # outputs normalisés
    + ["u"]
    + ["split"]
)
df = df[COLS_ORDER]

# Dataset
df.to_csv("wave_beam_dataset.csv", index=False)
print(f"Dataset sauvegardé : wave_beam_dataset.csv")
print(f"  {len(df):,} lignes × {len(df.columns)} colonnes")

# Stats de normalisation — à recharger dans les fichiers 2 et 3
norm_stats.to_csv("norm_stats.csv")
print(f"Stats de normalisation sauvegardées : norm_stats.csv")

Dataset sauvegardé : wave_beam_dataset.csv
  965,300 lignes × 12 colonnes
Stats de normalisation sauvegardées : norm_stats.csv


In [10]:
import json

params_simulation = {
    "L"          : L,
    "Nx"         : Nx,
    "Nt"         : Nt,
    "t_end"      : t_end,
    "E": E,
    "rho":rho,
    "AMPLITUDES" : AMPLITUDES,
    "PULSATIONS" : PULSATIONS,
    "n_sims"     : n_sims,
    "INPUTS":INPUTS,
    "OUTPUTS":OUTPUTS,
}

with open("params_simulation.json", "w") as f:
    json.dump(params_simulation, f, indent=2)

print("Paramètres sauvegardés : params_simulation.json")

Paramètres sauvegardés : params_simulation.json
